# Qwen3.5-0.8B, Full Fine-Tune, for Arabic Diacritisation

**KAUST Academy / DiacriticS.** Full-parameter fine-tuning of `Qwen/Qwen3.5-0.8B` on the whole
`Misraj/Sadeed_Tashkeela` corpus, evaluated on `Misraj/SadeedDiac-25` (1,200 paragraphs, 600
Classical Arabic + 600 Modern Standard Arabic).

| | |
|---|---|
| **Base model** | `Qwen/Qwen3.5-0.8B` (about 751M params, ungated) |
| **Method** | full fine-tune, all parameters trainable, fp32 master weights |
| **Training data** | `Misraj/Sadeed_Tashkeela`, 1,042,693 rows, **one full epoch** |
| **Hardware** | 1x H100 80GB (Modal), 4h 48m measured, roughly $27 |
| **Shipped weights** | best-validation checkpoint at **step 4,070** of 16,293 |
| **Headline** | SadeedDiac-25 macro DER (no case ending): **92.28 zero-shot -> 2.58 tuned** |

### The question this run exists to answer

> Does a **smaller model, fully fine-tuned on the entire corpus**, beat the 4B LoRA adapter that
> was trained on 9.2% of it?

Yes, decisively: **2.58 against 8.48** macro DER without case endings, a 70% relative reduction,
at one fifth the parameter count. See section 10.

Everything except the model and the training method is held fixed against the 4B run: the same
prompt, the same metric functions, the same NFC handling, the same generation settings, the same
effective batch of 64, the same seed. Exactly two variables change.

| | 4B LoRA | 0.8B full FT (this notebook) |
|---|---|---|
| model | `Qwen/Qwen3.5-4B` | `Qwen/Qwen3.5-0.8B` |
| method | LoRA r=16 | full fine-tune |
| trainable params | 21,233,664 (0.50%) | about 751,000,000 (100%) |
| corpus seen | 96,000 rows (9.2%) | 1,042,693 rows (100%) |
| optimizer steps | 1,500 | 16,293 |

### How to read this notebook

Sections 1 to 8 are the training and evaluation pipeline; they need one 80GB GPU and gated
access to `Misraj/Sadeed_Tashkeela`. **Section 9 onward runs anywhere**, recomputing every result
table from the CSV files in `results/`.

This is a distilled, readable version of the production script that actually ran
(`Train-Qwen-0.8B-Full-FT/train_full_ft_qwen35_08b.py`, 2,721 lines). Every correctness fix is
preserved and commented; the Modal harness, the resume plumbing and the CLI parsing are not.

---
## 1. Environment

Identical pins to the 4B run, deliberately. `causal-conv1d` and `flash-linear-attention` supply
the fused kernels for Qwen3.5's Gated-DeltaNet layers; without them the model falls back to a
pure-PyTorch scan that is several times slower and warns exactly once. `causal-conv1d` compiles
from source, so the CUDA toolkit and the torch wheel must agree on a major version.

In [ ]:
!pip install -q "torch==2.13.0" --index-url https://download.pytorch.org/whl/cu130
!pip install -q "transformers==5.14.1" "datasets==5.0.0" "accelerate==1.12.0" \
                "jiwer==4.0.0" "pandas" "matplotlib"
!pip install -q "flash-linear-attention==0.4.0" "causal-conv1d==1.6.0" --no-build-isolation

In [ ]:
import os, re, gc, json, math, random, unicodedata, pathlib, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = pathlib.Path("results")
RESULTS.mkdir(exist_ok=True)

---
## 2. Configuration

`results/run_config.json` is the copy the training job itself wrote out, including which
checkpoint the saved weights actually came from.

In [ ]:
# ---------------------------------------------------------------- model
MODEL_ID        = "Qwen/Qwen3.5-0.8B"          # ungated
CACHE_DIR       = os.environ.get("HF_CACHE_DIR", "./hf_cache")
OUTPUT_DIR      = "./qwen3.5-0.8b-fullft-diacritization"
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "final_model")

# ---------------------------------------------------------------- data
TRAIN_DATASET   = "Misraj/Sadeed_Tashkeela"    # gated, manual approval
INPUT_COLUMN    = "input"
OUTPUT_COLUMN   = "output"
FILENAME_COLUMN = "filename"                   # provenance, drives the corpus analysis

BENCHMARK_DATASET     = "Misraj/SadeedDiac-25"
BENCHMARK_SPLIT       = "train"
BENCHMARK_TEXT_COLUMN = "output"

TRAIN_DATASET_REVISION     = "c10bcbb3b50dc96551f62c472389de666a8c1c4e"
BENCHMARK_DATASET_REVISION = "aa311213e44e4cab6cc3f2848daacd753adc1ce1"

# ---------------------------------------------------------------- optimisation
NUM_TRAIN_EPOCHS            = 1.0
PER_DEVICE_TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8         # effective batch 64, same as the 4B run
PER_DEVICE_EVAL_BATCH_SIZE  = 8

# 2e-5, NOT the LoRA run's 2e-4. LoRA adapters start at zero and need a large LR to move; every
# weight here is pretrained and 2e-4 would wreck them.
LEARNING_RATE     = 2e-5
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO      = 0.03
WEIGHT_DECAY      = 0.01
MAX_GRAD_NORM     = 1.0                 # the HF default, spelled out: full FT can and does spike
MAX_SEQ_LENGTH    = 1024

# OFF. Worth about 33% extra compute, inherited from a config where it was needed. It is not
# needed here: see section 6 for the memory budget.
GRADIENT_CHECKPOINTING = False

LOGGING_STEPS           = 20
N_EVALS                 = 20            # intervals are DERIVED from the real step count
SAVE_TOTAL_LIMIT        = 2
IN_TRAINING_EVAL_SUBSET = 1000

# ---------------------------------------------------------------- inference
INFER_BATCH_SIZE       = 8
MAX_NEW_CAP            = 1024
GEN_LEN_RATIO          = 2.2
TRAIN_EVAL_SAMPLE_SIZE = 500

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Byte-identical to the 4B run's prompt. This is what makes the two benchmark numbers comparable.
SYSTEM_PROMPT = (
    "أنت نظام متخصص في التشكيل الآلي للنصوص العربية. "
    "مهمتك إضافة الحركات (التشكيل) الصحيحة إلى النص العربي المُدخل دون تغيير الكلمات أو ترتيبها، "
    "مع مراعاة السياق النحوي والصرفي الكامل للجملة."
)

# Used only if the model has no chat template (for example a -Base checkpoint).
PROMPT_TMPL = SYSTEM_PROMPT + "\n\nالنص:\n{inp}\n\nالنص المشكل:\n"

---
## 3. Evaluation metrics

Two rates, each with and without the word-final case ending (i'rab):

- **DER** (Diacritic Error Rate): wrong diacritisable characters / diacritisable characters.
- **WER** (Word Error Rate): words not exactly right / words.

Predictions and references are aligned word by word on their **consonantal skeletons** with
`jiwer`, so a model that drops or invents a word is still scored against the right reference
words.

### Both scorers run, over the same predictions

The frozen scorer is byte-identical to the 4B script, which is the only thing that makes a
0.8B-versus-4B delta mean anything. But re-scoring the 4B predictions exposed four defects in it,
so the corrected scorer runs too. Generation is not repeated, so this costs CPU seconds and no
GPU time.

| | frozen scorer | corrected scorer |
|---|---|---|
| purpose | comparability with the 4B run's 6.96 | **the number to cite** |
| mark position | flat positional list, host letter discarded | per host letter |
| DER denominator | marks present, depends on the prediction | diacritisable characters in the reference |
| inserted words | skipped, so hallucination is free | counted as errors |
| mark class | `[U+064B-U+0652]`, misses dagger alef | `[U+064B-U+065F] + U+0670`, tatweel stripped |

**Never mix the two columns.** On the 4B predictions the macro DER without case endings is 6.96
frozen and 8.48 corrected: one run scored twice, not two runs.

In [ ]:
import jiwer

ARABIC_DIACRITICS   = re.compile(r'[ً-ْ]')              # U+064B..U+0652, the frozen mark class
_EASTERN_TO_WESTERN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_numerals(text: str) -> str:
    return text.translate(_EASTERN_TO_WESTERN)

def strip_citation_refs(text: str) -> str:
    """Removes trailing parenthetical volume/page refs like '(41 / 251)' that pepper the corpus."""
    text = re.sub(r'\(\s*\d+\s*/\s*\d+\s*\)', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def nfc(text: str) -> str:
    """Canonical Unicode ordering. Not cosmetic, and not optional.

    About 90% of SadeedDiac-25's gold rows store shadda BEFORE the vowel on the same letter; a
    tokenizer-based model emits the canonical order. Both render as the identical word, but the
    frozen scorer compares marks positionally and scores it as an error. Worth 2.4x on identical
    predictions: DER_noce 19.13 raw versus 7.62 normalised.
    """
    return unicodedata.normalize("NFC", text)

In [ ]:
# ================================================================ FROZEN scorer
# Byte-identical to the 4B script. Kept ONLY so the 0.8B-vs-4B delta is measured on one ruler.
# Do not "improve" anything in this block.

def strip_diacritics(text: str) -> str:
    return ARABIC_DIACRITICS.sub('', text)

def clean_and_tokenize(text: str) -> list:
    text = strip_citation_refs(normalize_numerals(text))
    return re.sub(r'[^\w\sً-ْ]', '', text).split()

def strip_case_ending(word: str) -> str:
    last_base_idx = None
    for i, ch in enumerate(word):
        if not ARABIC_DIACRITICS.match(ch):
            last_base_idx = i
    return word if last_base_idx is None else word[:last_base_idx + 1]

def _is_digit_only(word: str) -> bool:
    return strip_diacritics(word).isdigit()

def _align_words(predictions: list, references: list):
    """Inserted (hallucinated) words are skipped: defect 3."""
    for pred, ref in zip(predictions, references):
        pred_words, ref_words = clean_and_tokenize(pred), clean_and_tokenize(ref)
        alignment = jiwer.process_words(
            " ".join(strip_diacritics(w) for w in ref_words),
            " ".join(strip_diacritics(w) for w in pred_words),
        )
        for chunk in alignment.alignments[0]:
            if chunk.type == "equal":
                for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    yield ref_words[i], pred_words[chunk.hyp_start_idx + (i - chunk.ref_start_idx)]
            elif chunk.type in ("substitute", "delete"):
                for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    yield ref_words[i], None

def compute_der(predictions, references, ce=True) -> float:
    total_chars, wrong_chars = 0, 0
    for r_word, p_word in _align_words(predictions, references):
        if _is_digit_only(r_word):
            continue
        if ce is False:
            r_word = strip_case_ending(r_word)
            if p_word is not None:
                p_word = strip_case_ending(p_word)
        r_diacs = ARABIC_DIACRITICS.findall(r_word)
        if p_word is None:
            n = max(len(r_diacs), 1); total_chars += n; wrong_chars += n; continue
        p_diacs = ARABIC_DIACRITICS.findall(p_word)
        total_chars += max(len(r_diacs), len(p_diacs), 1)      # defect 2: prediction-dependent
        if r_diacs != p_diacs:
            mism  = sum(1 for a, b in zip(r_diacs, p_diacs) if a != b)
            mism += abs(len(r_diacs) - len(p_diacs))
            wrong_chars += max(mism, 1)
    return round(wrong_chars / total_chars * 100, 2) if total_chars else 0.0

def compute_wer(predictions, references, ce=True) -> float:
    total_words, wrong_words = 0, 0
    for r_word, p_word in _align_words(predictions, references):
        total_words += 1
        if p_word is None:
            wrong_words += 1; continue
        if ce is False:
            r_word, p_word = strip_case_ending(r_word), strip_case_ending(p_word)
        if r_word != p_word:
            wrong_words += 1
    return round(wrong_words / total_words * 100, 2) if total_words else 0.0

def compute_der_wer(predictions, references) -> dict:
    return {"DER_ce":   compute_der(predictions, references, ce=True),
            "DER_noce": compute_der(predictions, references, ce=False),
            "WER_ce":   compute_wer(predictions, references, ce=True),
            "WER_noce": compute_wer(predictions, references, ce=False)}

In [ ]:
# ================================================================ CORRECTED scorer
# Fixes the four defects. These are the numbers to cite.

CORR_DIACRITICS = re.compile(r'[ً-ٰٟ]')        # widened to include U+0670 dagger alef
CORR_LETTER     = re.compile(r'[ء-غف-يٱ-ۓ]')
TATWEEL         = "ـ"

def corr_strip_diacritics(text: str) -> str:
    """Skeleton, dropping tatweel too: 'الرحمـن' and 'الرحمن' would otherwise align as a
    substitution and the reference word would be scored as a deletion, a fabricated error."""
    return CORR_DIACRITICS.sub('', text).replace(TATWEEL, '')

def corr_clean_and_tokenize(text: str) -> list:
    text = unicodedata.normalize("NFC", text)          # normalise inside, not at the call site
    text = strip_citation_refs(normalize_numerals(text)).replace(TATWEEL, '')
    return re.sub(r'[^\w\sً-ٰٟ]', '', text).split()

def corr_segment(word: str) -> list:
    """Splits a word into [(base_char, marks), ...]: the fix for defect 1. Marks are sorted so
    two canonically equivalent orderings of a cluster compare equal."""
    units = []
    for ch in unicodedata.normalize("NFC", word):
        if CORR_DIACRITICS.match(ch):
            if units:
                units[-1][1] += ch
        else:
            units.append([ch, ""])
    return [(base, "".join(sorted(marks))) for base, marks in units]

def corr_scorable(units: list, ce: bool) -> list:
    """Diacritisable units, dropping the final one when ce=False. Dropping the unit outright is
    Fadel's 'excluding last character', the convention the published numbers use."""
    idx = [i for i, (base, _) in enumerate(units) if CORR_LETTER.match(base)]
    if not ce and idx:
        idx = idx[:-1]
    return [units[i] for i in idx]

def corr_is_digit_only(word: str) -> bool:
    return corr_strip_diacritics(word).isdigit()

def corr_align_words(predictions, references) -> list:
    """Insertions are EMITTED as (None, pred): the fix for defect 3."""
    pairs = []
    for pred, ref in zip(predictions, references):
        pred_words = [w for w in corr_clean_and_tokenize(pred) if corr_strip_diacritics(w)]
        ref_words  = [w for w in corr_clean_and_tokenize(ref)  if corr_strip_diacritics(w)]
        alignment = jiwer.process_words(
            " ".join(corr_strip_diacritics(w) for w in ref_words),
            " ".join(corr_strip_diacritics(w) for w in pred_words),
        )
        for chunk in alignment.alignments[0]:
            if chunk.type == "equal":
                for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    pairs.append((ref_words[i],
                                  pred_words[chunk.hyp_start_idx + (i - chunk.ref_start_idx)]))
            elif chunk.type == "substitute":
                pairs += [(ref_words[i], None) for i in range(chunk.ref_start_idx, chunk.ref_end_idx)]
                pairs += [(None, pred_words[j]) for j in range(chunk.hyp_start_idx, chunk.hyp_end_idx)]
            elif chunk.type == "delete":
                pairs += [(ref_words[i], None) for i in range(chunk.ref_start_idx, chunk.ref_end_idx)]
            elif chunk.type == "insert":
                pairs += [(None, pred_words[j]) for j in range(chunk.hyp_start_idx, chunk.hyp_end_idx)]
    return pairs

def corr_compute_der(pairs, ce=True) -> float:
    """Denominator is diacritisable REFERENCE characters, fixed per test set: the fix for
    defect 2. Insertions add to both terms."""
    total, wrong = 0, 0
    for r_word, p_word in pairs:
        if r_word is None:                                     # insertion
            if p_word is None or corr_is_digit_only(p_word):
                continue
            n = len(corr_scorable(corr_segment(p_word), ce)); total += n; wrong += n; continue
        if corr_is_digit_only(r_word):
            continue
        r_units = corr_scorable(corr_segment(r_word), ce)
        total += len(r_units)
        if p_word is None:
            wrong += len(r_units); continue
        p_units = corr_scorable(corr_segment(p_word), ce)
        for i, (r_base, r_marks) in enumerate(r_units):
            if i >= len(p_units) or p_units[i] != (r_base, r_marks):
                wrong += 1
    return round(wrong / total * 100, 2) if total else float("nan")

def corr_compute_wer(pairs, ce=True) -> float:
    total, wrong = 0, 0
    for r_word, p_word in pairs:
        total += 1
        if r_word is None or p_word is None:
            wrong += 1; continue
        if corr_scorable(corr_segment(r_word), ce) != corr_scorable(corr_segment(p_word), ce):
            wrong += 1
    return round(wrong / total * 100, 2) if total else float("nan")

def compute_der_wer_corrected(predictions, references) -> dict:
    """Aligns once and reuses the pairs for all four numbers."""
    pairs = corr_align_words(predictions, references)
    return {"DER_ce_corr":   corr_compute_der(pairs, ce=True),
            "DER_noce_corr": corr_compute_der(pairs, ce=False),
            "WER_ce_corr":   corr_compute_wer(pairs, ce=True),
            "WER_noce_corr": corr_compute_wer(pairs, ce=False)}

---
## 4. Data

`Misraj/Sadeed_Tashkeela`: 1,042,698 train rows and 2,485 test rows of `(undiacritised,
diacritised)` pairs, **gated**, so an approved Hugging Face token is required.

Two guards are load bearing:

1. **Null rows.** Exactly 1 row of 1,042,698 has a null text column despite the corpus being
   documented as pre-cleaned. It killed a run 12 minutes in with the GPU already rented, and a
   200-row smoke test cannot find it. Rows are dropped, not coerced to `""`, which would train
   the model to answer with nothing.
2. **Pinned revisions.** Passing `revision=` is what makes the test set fixed.

A full pass over the corpus is the only thing that finds corpus defects, and it is far cheaper to
do it on CPU before renting the GPU.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset

login(token=os.environ["HF_TOKEN"])       # never hardcode the token in the file

In [ ]:
def _drop_unusable_rows(ds, name, columns):
    before = len(ds)
    ds = ds.filter(lambda ex: all(isinstance(ex[c], str) and ex[c].strip() for c in columns),
                   num_proc=min(8, os.cpu_count() or 1), desc=f"filtering {name}")
    if before - len(ds):
        print(f"[INFO] {name}: dropped {before - len(ds)} unusable row(s) of {before}")
    return ds


def load_training_data(include=None, exclude=None):
    """`include`/`exclude` are regexes over the source filename. They are the mechanism for
    building a domain-balanced rung: see section 10.2. No default mix is hardcoded, because
    guessing one without reading the composition would be worse than not offering it."""
    ds    = load_dataset(TRAIN_DATASET, revision=TRAIN_DATASET_REVISION, cache_dir=CACHE_DIR)
    train = _drop_unusable_rows(ds["train"], "train", [INPUT_COLUMN, OUTPUT_COLUMN])
    test  = _drop_unusable_rows(ds["test"],  "test",  [INPUT_COLUMN, OUTPUT_COLUMN])
    for pattern, keep in ((include, True), (exclude, False)):
        if pattern:
            rx = re.compile(pattern)
            before = len(train)
            train = train.filter(lambda ex: bool(rx.search(ex[FILENAME_COLUMN] or "")) is keep)
            print(f"[INFO] filename filter {'include' if keep else 'exclude'} {pattern!r}: "
                  f"{before:,} -> {len(train):,} rows")
    print(f"train {len(train):,} rows   test {len(test):,} rows")
    return train, test


def load_benchmark_data():
    ds = load_dataset(BENCHMARK_DATASET, revision=BENCHMARK_DATASET_REVISION,
                      split=BENCHMARK_SPLIT, cache_dir=CACHE_DIR)
    print(f"benchmark {len(ds):,} rows   columns {ds.column_names}")
    return ds


def benchmark_pairs(dataset):
    """SadeedDiac-25 ships gold diacritised text only, so the input is derived by stripping the
    marks. That guarantees input and reference share a skeleton."""
    return [(strip_diacritics(r[BENCHMARK_TEXT_COLUMN]), r[BENCHMARK_TEXT_COLUMN])
            for r in dataset]


def benchmark_domains(dataset):
    """CA versus MSA labels. This split is the finding, not a detail."""
    for col in ("is_msa", "domain", "source", "category"):
        if col in dataset.column_names:
            vals = dataset[col]
            if col == "is_msa":
                return ["MSA" if bool(v) else "CA" for v in vals]
            return ["MSA" if str(v).upper().startswith("MSA") else "CA" for v in vals]
    return None

### Corpus composition, before deciding this is the right run

The 4B run's central finding is that fine-tuning **creates** a CA/MSA asymmetry rather than
inheriting one, because `Sadeed_Tashkeela` is Tashkeela-derived and therefore Classical-heavy. A
full epoch of that same corpus is a full epoch of more Classical Arabic. Reading the composition
first is what tells you whether the plain full-corpus run is even the right next run.

In [ ]:
def analyze_corpus(train_raw, top=25):
    """Row counts per source file. Writes results/corpus_composition.csv."""
    counts = pd.Series(train_raw[FILENAME_COLUMN]).value_counts()
    counts.rename_axis("filename").rename("rows").to_csv(RESULTS / "corpus_composition.csv")
    total = int(counts.sum())
    print(f"{len(counts)} source files, {total:,} rows")
    print(f"top {top} account for {counts.head(top).sum() / total:.1%} of the corpus\n")
    return counts.head(top).to_frame("rows").assign(share=lambda d: (d.rows / total * 100).round(2))


train_raw, test_raw = load_training_data()
benchmark_raw       = load_benchmark_data()
analyze_corpus(train_raw)

---
## 5. Prompt construction

Identical to the 4B run. Two settings are not optional:

- **`enable_thinking=False`.** At its default the template ends the prompt at an *unclosed*
  `<think>\n`, and training through that teaches the model to answer inside a reasoning block.
- **`</think>` is not a special token** in this tokenizer, so `skip_special_tokens=True` leaves
  it in the decoded string as literal text, where it would be scored as content.

The prompt mode is **detected**, not assumed. A `-Base` checkpoint has no chat template, and the
script falls back to the plain `PROMPT_TMPL` and says so, because that changes the prompt as well
as the model and the 4B comparison then no longer isolates two variables.

In [ ]:
_CHAT_MODE            = None      # True = chat template, False = plain PROMPT_TMPL
_SUPPORTS_SYSTEM_ROLE = None

def detect_prompt_mode(tokenizer):
    global _CHAT_MODE, _SUPPORTS_SYSTEM_ROLE
    if _CHAT_MODE is not None:
        return _CHAT_MODE
    if not getattr(tokenizer, "chat_template", None):
        _CHAT_MODE = False
        print("[WARN] no chat template; falling back to the plain PROMPT_TMPL. This changes the "
              "prompt as well as the model, so the 4B comparison no longer isolates 2 variables.")
        return _CHAT_MODE
    _CHAT_MODE = True
    try:
        tokenizer.apply_chat_template(
            [{"role": "system", "content": "x"}, {"role": "user", "content": "y"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
        _SUPPORTS_SYSTEM_ROLE = True
    except Exception:
        _SUPPORTS_SYSTEM_ROLE = False
        print("[INFO] no system role; folding the instruction into the user turn")
    return _CHAT_MODE


def build_prompt_messages(user_text: str) -> list:
    if _SUPPORTS_SYSTEM_ROLE:
        return [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_text}]
    return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_text}"}]


def render_prompt(tokenizer, user_text: str) -> str:
    if not detect_prompt_mode(tokenizer):
        return PROMPT_TMPL.format(inp=user_text)
    return tokenizer.apply_chat_template(
        build_prompt_messages(user_text),
        tokenize=False, add_generation_prompt=True, enable_thinking=False)


def assert_prompt_sane(tokenizer):
    rendered = render_prompt(tokenizer, "نص")
    if "<think>" in rendered and "</think>" not in rendered:
        raise RuntimeError("prompt ends inside an unclosed <think> block; enable_thinking is not "
                           "taking effect on this transformers version")
    print("[OK] prompt closes its reasoning block")


_THINK_RE          = re.compile(r"^\s*(?:<think>)?.*?</think>\s*", re.DOTALL)
_ECHOED_HEADER_RE  = re.compile(r"^\s*النص المشكل\s*:?\s*", re.MULTILINE)

def clean_output(text: str, chat_mode: bool = True) -> str:
    """Two modes, because the two prompt formats fail differently.

    chat mode: strip a leading reasoning block only. Do NOT truncate at a blank line, because
    this model emits <|im_end|> and truncating would amputate any target that legitimately
    contains one.

    plain mode (-Base): additionally strip an echoed prompt header and truncate at the first
    blank line, because a base model has no turn-end token and generates to the cap forever,
    usually by inventing a new document.
    """
    out = _THINK_RE.sub("", text.strip()).strip()
    if not chat_mode:
        out = _ECHOED_HEADER_RE.sub("", out).strip().split("\n\n")[0].strip()
    return out


def generation_eos_ids(tokenizer) -> list:
    """The repo ships no generation_config.json and its config EOS is <|endoftext|> (248044), not
    the <|im_end|> (248046) that ends a chat turn. Left alone, every generation runs to the cap
    and every tail is scored as a deletion."""
    ids = {tokenizer.eos_token_id}
    for tok in ("<|im_end|>", "<|endoftext|>"):
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid >= 0:
            ids.add(tid)
    return sorted(i for i in ids if i is not None)

---
## 6. Model loading, and the one defect that would silently ruin the run

### fp32 master weights

This is the failure most likely to produce a run that completes, logs a plausible falling loss,
and has learned a fraction of what it should have.

Load the model in bf16 and torch AdamW allocates `exp_avg` and `exp_avg_sq` with `zeros_like(p)`,
so the optimizer state is bf16 too and there are **no fp32 master weights**. In bf16,
`1.0 + 1e-4 == 1.0` exactly, and at `lr=2e-5` that is the scale of a typical update. Most of
training rounds away.

It has to be an **assertion** rather than a comment, because transformers 5.x resolves an
unspecified `dtype` from the checkpoint config, and `Qwen3.5-0.8B/config.json` says
`"dtype": "bfloat16"`. Say nothing and you get the broken path by default.

At 751M params the fp32 state is affordable, so this trains with the stock `Trainer`: fp32
weights, `bf16=True` autocast for the forward and backward. No DeepSpeed, no ZeRO, no sharding.

```
fp32 weights 3.0GB + fp32 grads 3.0GB + AdamW m,v 6.0GB = 12.0GB
```

### Why gradient checkpointing is off, and why the batch stays at 64

Memory budget on an 80GB H100 at batch 8 x 1024 tokens:

| | |
|---|---|
| fp32 weights + grads + AdamW m,v | 12.0 GB |
| autocast bf16 weight copies | ~1.5 GB |
| **logits + fp32 cross-entropy upcast + gradient** | **~20.0 GB** |
| activations, 24 layers, unchecked | ~6.0 GB |
| **total** | **~40 GB of 80** |

Checkpointing costs about 33% extra compute and buys nothing here, so it is the single largest
free speed-up available.

The batch stays at an effective 64 for two reasons. First, it is what the 4B LoRA run used, and
keeping the optimization geometry fixed is what makes the two benchmark numbers mean something
side by side. Second, the ceiling is the **logit tensor, not the parameters**: `vocab_size` is
248,320, so logits are `batch x seq x 248,320` and cross-entropy upcasts them to fp32. A batch of
16 would put peak memory near 65GB on the longest length-grouped batches, and the length-grouped
sampler puts the longest batch *first* in every megabatch, so an OOM would not wait politely
until the end of the run.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def assert_fast_path():
    import importlib
    for mod in ("fla", "causal_conv1d"):
        if importlib.util.find_spec(mod) is None:
            raise RuntimeError(f"{mod} is not installed; linear-attention layers will crawl")
    print("[OK] linear-attention fast path available")


def assert_text_only(model):
    """No vision tower, no MTP head in the causal-LM forward path. If either loaded, the
    parameter count and the memory budget above are both wrong."""
    stray = sorted({n.split(".")[0] for n, _ in model.named_parameters()} & {"visual", "mtp"})
    if stray:
        raise RuntimeError(f"unexpected towers loaded: {stray}; this is not the text-only class")
    print("[OK] text-only model class")


def assert_master_dtype(model):
    """THE assertion. Trainable params must be fp32 or the updates round away under bf16."""
    dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
    if dtypes != {torch.float32}:
        raise RuntimeError(
            f"trainable params are {dtypes}, not float32. torch AdamW would allocate its state "
            f"with zeros_like(p) and there would be no fp32 master weights; at lr={LEARNING_RATE} "
            f"most updates round away and the run silently learns almost nothing.")
    print("[OK] fp32 master weights")


def report_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"all params       {total:>15,}")
    print(f"trainable params {train:>15,}   {train / total:.4%}")
    return total, train

In [ ]:
def load_model_for_training():
    assert_fast_path()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # dtype=torch.float32 is explicit and load bearing. Omit it and transformers 5.x reads
    # "bfloat16" from the checkpoint config and hands back the broken path.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, cache_dir=CACHE_DIR, dtype=torch.float32, device_map=None)
    model.config.use_cache = False

    assert_text_only(model)
    assert_master_dtype(model)
    report_parameters(model)
    assert_prompt_sane(tokenizer)
    return model, tokenizer


model, tokenizer = load_model_for_training()

---
## 7. Training

### Label masking

Prompt tokens are masked with `-100` so the loss lands on the diacritised answer only. A loss near
0.0 at step 1 means the mask is inverted, and a callback raises on exactly that at the first
logged step rather than at the end of a five-hour run.

### Length-grouped batching, and the hang it causes

Lengths run about p50 380 / p99 900 against `MAX_SEQ_LENGTH=1024`, and the collator pads each
batch to its own maximum, so random sampling burns 35 to 42% of every pass on padding. Grouping
drops that to about 0.5%.

The catch: `Trainer._get_train_sampler` reads `train_dataset["length"]`, which under `datasets`
5.x returns a **lazy Arrow Column**, not a list. `LengthGroupedSampler` stores it as-is and sorts
every row through it, and `.map()` writes 1,000-row chunks, so each lookup pays a chunk search.
At 1M rows that is about 6 minutes sitting at step 0 looking exactly like a hang. Materialising
the column costs about 26 seconds.

Use `list(dataset["length"])`, **not** `dataset.data.column("length").to_pylist()`: the latter
reads the underlying Arrow table and ignores the indices map `.filter()` installs, so after the
null-row filter it returns the wrong row count, silently misaligned with the dataset.

### Two fixes this run added over the 4B script

**Deterministic eval ordering.** transformers 5.14.1 drives *both* samplers off the single
`train_sampling_strategy` field, so `group_by_length` also makes `_get_eval_sampler` return a
`LengthGroupedSampler` built with `generator=None`, giving a different dev ordering at every
evaluation. `eval_loss` is a sample-weighted mean of per-batch token-mean losses, so regrouping
moves the number for no modelling reason. That matters more here than at 4B, because `eval_loss`
selects the final weights across 16,293 steps rather than 1,500.

**Derived eval and save intervals, with `save_steps == eval_steps`.** This one is subtle and it
silently ships the wrong weights. In transformers 5.14.1, `Trainer._determine_best_metric` runs at
every eval and updates `best_metric` and `best_global_step` with no requirement that a checkpoint
exist there, while `_save_checkpoint` adopts the best only `if os.path.exists(best_checkpoint_dir)`.
Evaluate more often than you checkpoint and a minimum at an unsaved step pins `best_metric`
permanently, is never adoptable, and leaves `best_model_checkpoint` as `None`. The
`load_best_model_at_end` guard then skips the load entirely: the run finishes, saves the
**final-step** weights as if they were selected, and logs nothing about it.

That is not hypothetical. The 4B run bottomed at step 400 of 1,500, so an early minimum is the
expected shape here. `SAVE_TOTAL_LIMIT` already bounds *resident* checkpoints however often they
are written, so saving rarely bought no storage, only write time.

In [ ]:
from transformers import (Trainer, TrainingArguments, TrainerCallback,
                          DataCollatorForSeq2Seq)
from transformers.trainer_pt_utils import LengthGroupedSampler
from torch.utils.data import SequentialSampler


def build_tokenizer_fn(tokenizer):
    eos = tokenizer.eos_token or ""
    def tokenize(ex):
        prompt_ids = tokenizer(render_prompt(tokenizer, ex[INPUT_COLUMN]),
                               add_special_tokens=False)["input_ids"]
        answer_ids = tokenizer(ex[OUTPUT_COLUMN] + eos, add_special_tokens=False)["input_ids"]
        input_ids  = (prompt_ids + answer_ids)[:MAX_SEQ_LENGTH]
        labels     = ([-100] * len(prompt_ids) + answer_ids)[:MAX_SEQ_LENGTH]
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
                "labels": labels, "length": len(input_ids)}
    return tokenize


class _CollatorDroppingExtras:
    """`length` must reach the dataset for the sampler, so it also reaches the collator.
    Qwen3_5ForCausalLM.forward has **kwargs and would swallow it silently. Strip explicitly."""
    MODEL_KEYS = ("input_ids", "attention_mask", "labels")
    def __init__(self, base): self.base = base
    def __call__(self, features):
        return self.base([{k: f[k] for k in self.MODEL_KEYS if k in f} for f in features])
    def __getattr__(self, item): return getattr(self.base, item)


def _length_grouping_kwargs(cls) -> dict:
    """transformers 5.x renamed `group_by_length` to `train_sampling_strategy`; handle both."""
    fields = {f.name for f in dataclasses.fields(cls)}
    if "train_sampling_strategy" in fields:
        kw = {"train_sampling_strategy": "group_by_length"}
    elif "group_by_length" in fields:
        kw = {"group_by_length": True}
    else:
        return {}
    if "length_column_name" in fields:
        kw["length_column_name"] = "length"
    kw["remove_unused_columns"] = False
    return kw


class FullFTTrainer(Trainer):
    """Materialised train lengths, plus deterministic sequential eval ordering."""
    def __init__(self, *args, train_lengths=None, **kwargs):
        self._train_lengths = train_lengths
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, *args, **kwargs):
        grouped = (getattr(self.args, "train_sampling_strategy", None) == "group_by_length"
                   or getattr(self.args, "group_by_length", False))
        if self._train_lengths is None or not grouped:
            return super()._get_train_sampler(*args, **kwargs)
        proc = getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)
        return LengthGroupedSampler(
            self.args.train_batch_size * self.args.gradient_accumulation_steps,
            lengths=self._train_lengths,
            model_input_name=proc.model_input_names[0] if proc is not None else None)

    def _get_eval_sampler(self, eval_dataset=None, *args, **kwargs):
        if eval_dataset is None:
            return None
        return SequentialSampler(eval_dataset) if self.args.world_size <= 1 else None


class LossSanityCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss")
        if loss is None or getattr(self, "_seen", False):
            return
        self._seen = True
        if not math.isfinite(loss):
            raise RuntimeError(f"first logged loss is {loss}; stop and check the data")
        if loss < 0.05:
            raise RuntimeError(f"first logged loss is {loss:.4f}, near zero. The model is being "
                               f"scored on tokens it was handed: the label mask is inverted.")
        print(f"[OK] first logged loss {loss:.4f} is in a sane range")

In [ ]:
def schedule_for(total_steps: int, n_evals: int = N_EVALS) -> dict:
    """Derives eval/save intervals from the REAL step count, and forces save_steps == eval_steps.

    Fixed intervals do not survive a change of run length: EVAL_STEPS=200 over a full epoch here
    would be 81 evaluations and 81 checkpoints of about 9GB each. And evaluating more often than
    checkpointing can pin `best_metric` at a step that has no checkpoint, which makes
    load_best_model_at_end silently ship the final weights instead. See the note above.
    """
    step = max(1, total_steps // max(1, n_evals))
    return {"eval_steps": step, "save_steps": step}


def assert_best_model_selected(trainer):
    """Re-checks the outcome after training, so the saved model states which weights it holds."""
    st = trainer.state
    if st.best_model_checkpoint is None:
        print(f"[WARN] no best checkpoint was adopted; the saved weights are the FINAL step "
              f"({st.global_step}), not the best eval_loss step ({st.best_global_step}).")
        return {"weights_are": "final_step", "final_global_step": st.global_step}
    print(f"[OK] shipping best checkpoint {st.best_model_checkpoint} "
          f"(step {st.best_global_step}, eval_loss {st.best_metric:.10f}) "
          f"of {st.global_step} total steps")
    return {"weights_are": "best_checkpoint",
            "best_model_checkpoint": st.best_model_checkpoint,
            "best_eval_loss": st.best_metric,
            "best_global_step": st.best_global_step,
            "final_global_step": st.global_step}

In [ ]:
def train_model(model, tokenizer, train_raw, test_raw):
    tok_fn = build_tokenizer_fn(tokenizer)

    train_ds = train_raw.map(tok_fn, remove_columns=train_raw.column_names,
                             num_proc=min(8, os.cpu_count() or 1), desc="tokenising train")
    eval_ds  = test_raw.select(range(min(IN_TRAINING_EVAL_SUBSET, len(test_raw)))) \
                       .map(tok_fn, remove_columns=test_raw.column_names, desc="tokenising eval")

    print("materialising the length column ...")
    train_lengths = list(train_ds["length"])       # about 26s, saves ~6 min of apparent hang

    effective_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    total_steps     = math.ceil(len(train_ds) / effective_batch * NUM_TRAIN_EPOCHS)
    sched           = schedule_for(total_steps)
    print(f"{total_steps:,} optimizer steps, eval and save every {sched['eval_steps']:,}")

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps", save_strategy="steps", **sched,
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False,
        bf16=True,                              # autocast only; master weights stay fp32
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        seed=RANDOM_SEED, report_to="none",
        **_length_grouping_kwargs(TrainingArguments),
    )

    trainer = FullFTTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        processing_class=tokenizer,
        data_collator=_CollatorDroppingExtras(
            DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)),
        callbacks=[LossSanityCallback()],
        train_lengths=train_lengths,
    )
    assert_master_dtype(trainer.model)          # re-check AFTER Trainer has touched the model
    trainer.train()
    return trainer, total_steps


trainer, total_steps = train_model(model, tokenizer, train_raw, test_raw)

In [ ]:
provenance = assert_best_model_selected(trainer)

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

with open(os.path.join(FINAL_MODEL_DIR, "run_config.json"), "w") as f:
    json.dump({
        "base_model": MODEL_ID, "method": "full_fine_tune", "chat_mode": bool(_CHAT_MODE),
        "checkpoint_provenance": provenance,          # states WHICH weights this folder holds
        "epochs": NUM_TRAIN_EPOCHS, "planned_total_steps": total_steps,
        "lr": LEARNING_RATE, "lr_scheduler": LR_SCHEDULER_TYPE,
        "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM, "max_seq_length": MAX_SEQ_LENGTH,
        "effective_batch": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "master_dtype": "float32", "autocast": "bfloat16", "seed": RANDOM_SEED,
        "train_dataset": TRAIN_DATASET,
        "train_dataset_revision": TRAIN_DATASET_REVISION,
        "benchmark_dataset_revision": BENCHMARK_DATASET_REVISION,
        "enable_thinking": False,
    }, f, indent=2)

pd.DataFrame(trainer.state.log_history).to_csv(RESULTS / "training_log.csv", index=False)
print("model saved to", FINAL_MODEL_DIR)

---
## 8. Inference and scoring

Greedy decoding, left padding, and `max_new_tokens` scaled to the input rather than fixed. A flat
cap truncates real answers on this corpus, and every truncated tail is scored as a run of
deletions, so a fixed cap fabricates errors.

Truncation is counted **per row**, not per batch. `generate()` returns one rectangular tensor
whose width is set by the longest-running row, so comparing `out.shape[1]` to `max_new_tokens`
would mark all 8 rows of a batch as truncated whenever one of them was, and it would do so
precisely on the batches where a real truncation happened.

In [ ]:
@torch.no_grad()
def run_inference(model, tokenizer, pairs, name):
    """Returns (predictions, n_truncated)."""
    prev_pad, prev_trunc = tokenizer.padding_side, tokenizer.truncation_side
    tokenizer.padding_side    = "left"    # right padding corrupts batched generation
    # Truncate from the LEFT: the default cuts the END of the rendered template, i.e. the
    # "<|im_start|>assistant" part that tells the model to answer. The model would then continue
    # the user's text instead of diacritising it, and the row would look like a bad prediction
    # rather than a bug.
    tokenizer.truncation_side = "left"
    model.eval()
    eos_ids = generation_eos_ids(tokenizer)

    preds     = [None] * len(pairs)
    order     = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]))
    truncated = 0
    try:
        for start in range(0, len(order), INFER_BATCH_SIZE):
            idxs    = order[start:start + INFER_BATCH_SIZE]
            prompts = [render_prompt(tokenizer, pairs[i][0]) for i in idxs]
            # add_special_tokens=False: the chat template already emitted every special token
            # this prompt needs, as text.
            enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_SEQ_LENGTH, add_special_tokens=False).to(model.device)
            in_len  = enc["input_ids"].shape[1]
            out_ids = model.generate(
                **enc, max_new_tokens=min(int(in_len * GEN_LEN_RATIO) + 32, MAX_NEW_CAP),
                do_sample=False, num_beams=1,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids)
            gen_only = out_ids[:, in_len:]

            # A row that stopped early is right-padded with pad_token_id, which is in eos_ids,
            # so "contains no eos id" is exactly "ran to the cap".
            eos_t   = torch.tensor(eos_ids, device=gen_only.device)
            hit_eos = (gen_only.unsqueeze(-1) == eos_t).any(-1).any(-1)
            truncated += int((~hit_eos).sum().item())

            for i, text in zip(idxs, tokenizer.batch_decode(gen_only, skip_special_tokens=True)):
                preds[i] = clean_output(text, chat_mode=bool(_CHAT_MODE))
            if start % (INFER_BATCH_SIZE * 25) == 0:
                print(f"  {name}: {sum(p is not None for p in preds)}/{len(pairs)}")
    finally:
        tokenizer.padding_side, tokenizer.truncation_side = prev_pad, prev_trunc

    if truncated:
        print(f"[WARN] {truncated}/{len(pairs)} generations hit max_new_tokens; "
              f"truncated tails score as deletions and fabricate errors")
    return preds, truncated

In [ ]:
def score_predictions(name, references, predictions, truncated=0):
    """Scores one split with BOTH scorers, from ONE generation pass."""
    refs  = [nfc(r) for r in references]
    preds = [nfc(p) for p in predictions]
    row = {"split": name, "n_examples": len(refs),
           "n_empty_predictions": sum(1 for p in preds if not p.strip()),
           "n_truncated": truncated}
    row.update(compute_der_wer_corrected(preds, refs))   # cite these
    row.update(compute_der_wer(preds, refs))             # 4B comparability only
    return row


def evaluate_benchmark(model, tokenizer, benchmark_raw, label="ft"):
    """Generate SadeedDiac-25 ONCE, score CA / MSA / pooled natively, add the macro mean."""
    name    = f"{label}_benchmark_sadeeddiac25"
    pairs   = benchmark_pairs(benchmark_raw)
    domains = benchmark_domains(benchmark_raw)
    refs    = [p[1] for p in pairs]

    predictions, truncated = run_inference(model, tokenizer, pairs, name)
    pd.DataFrame({"domain": domains or "", "input": [p[0] for p in pairs],
                  "reference": refs, "prediction": predictions}) \
      .to_csv(RESULTS / f"predictions_sadeeddiac25_{label}.csv", index=False)

    rows = [score_predictions(f"{name}_pooled", refs, predictions, truncated)]
    if domains:
        for dom in ("CA", "MSA"):
            idx = [i for i, d in enumerate(domains) if d == dom]
            # Truncations are not attributed per domain: run_inference counts per batch and
            # batches mix domains. Reported on the pooled row only, so it is never wrong.
            rows.append(score_predictions(f"{name}_{dom}", [refs[i] for i in idx],
                                          [predictions[i] for i in idx], truncated=0))
        ca, msa = rows[-2], rows[-1]
        macro = {"split": f"{name}_MACRO_MEAN", "n_examples": ca["n_examples"] + msa["n_examples"],
                 "n_empty_predictions": ca["n_empty_predictions"] + msa["n_empty_predictions"],
                 "n_truncated": truncated}
        for m in ca:
            if m.startswith(("DER", "WER")):
                macro[m] = round((ca[m] + msa[m]) / 2, 2)
        rows.append(macro)

        print(f"\n  MACRO DER_noce, CORRECTED = {macro['DER_noce_corr']}   "
              f"CA {ca['DER_noce_corr']} / MSA {msa['DER_noce_corr']}")
        print("      ^ cite this. Compares to the 4B run's 8.48, and to corrected numbers only.")
        print(f"  MACRO DER_noce, frozen    = {macro['DER_noce']}   "
              f"CA {ca['DER_noce']} / MSA {msa['DER_noce']}")
        print("      ^ compares to the 4B run's 6.96. Comparison only, do not put it in a table.")

        # Judged on the CORRECTED numbers: the frozen scorer understates MSA specifically,
        # because MSA is where the model paraphrases and insertions were free.
        if msa["DER_noce_corr"] > 2 * max(ca["DER_noce_corr"], 0.01):
            print("\n  [NOTE] MSA error is >2x CA error, the same asymmetry the 4B run showed. "
                  "Sadeed_Tashkeela is Classical-heavy, so the model over-applies CA conventions "
                  "to MSA. More steps on this corpus will not fix it; a balanced mix might.")
    return rows

In [ ]:
rows = evaluate_benchmark(trainer.model, tokenizer, benchmark_raw, label="ft")

for split_name, raw in (("ft_test", test_raw),
                        ("ft_train", train_raw.shuffle(seed=RANDOM_SEED)
                                              .select(range(TRAIN_EVAL_SAMPLE_SIZE)))):
    p = [(r[INPUT_COLUMN], r[OUTPUT_COLUMN]) for r in raw]
    preds, trunc = run_inference(trainer.model, tokenizer, p, split_name)
    rows.append(score_predictions(split_name, [x[1] for x in p], preds, trunc))

pd.DataFrame(rows).to_csv(RESULTS / "metrics_finetuned_fullft.csv", index=False)
pd.DataFrame(rows)

### The zero-shot baseline

The only honest "before" for this exact prompt and scoring convention. The `label` argument is the
guard: prediction files are namespaced by it, and reusing `ft` for the base model would resume the
fine-tuned model's cached predictions and report them as the baseline, which is indistinguishable
from "fine-tuning changed nothing".

In [ ]:
del model, trainer
gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, cache_dir=CACHE_DIR, dtype=torch.bfloat16, device_map="auto")

base_rows = evaluate_benchmark(base_model, tokenizer, benchmark_raw, label="base")
pd.DataFrame(base_rows).to_csv(RESULTS / "metrics_baseline_zeroshot.csv", index=False)
pd.DataFrame(base_rows)

---
## 9. Results

**Everything below runs without a GPU**, from the CSV files in `results/`.

In [ ]:
import pathlib, json
import pandas as pd
RESULTS = pathlib.Path("results")

ft   = pd.read_csv(RESULTS / "metrics_finetuned_fullft.csv")
base = pd.read_csv(RESULTS / "metrics_baseline_zeroshot.csv")
log  = pd.read_csv(RESULTS / "training_log.csv")
cfg  = json.loads((RESULTS / "run_config.json").read_text())

cite = ["split", "n_examples", "n_truncated",
        "DER_ce_corr", "DER_noce_corr", "WER_ce_corr", "WER_noce_corr"]
print("FINE-TUNED (corrected scorer)")
print(ft[cite].to_string(index=False))
print("\nZERO-SHOT BASELINE (corrected scorer)")
print(base[cite].to_string(index=False))
cfg["checkpoint_provenance"]

### 9.1 Headline

| split | n | DER w/ CE | DER w/o CE | WER w/ CE | WER w/o CE |
|---|---|---|---|---|---|
| Sadeed_Tashkeela train (sample) | 500 | 0.94 | 0.83 | 2.28 | 1.47 |
| Sadeed_Tashkeela test | 2,485 | 2.52 | 1.91 | 5.62 | 2.75 |
| **SadeedDiac-25 CA** | 600 | 1.33 | 0.73 | 4.28 | 1.47 |
| **SadeedDiac-25 MSA** | 600 | 4.79 | 4.43 | 10.40 | 6.78 |
| **SadeedDiac-25 macro mean** | 1,200 | **3.06** | **2.58** | **7.34** | **4.12** |
| Qwen3.5-0.8B zero-shot, macro | 1,200 | 92.04 | 92.28 | 99.65 | 99.59 |

Use the **macro mean**, not the pooled number. Pooling is diacritic-weighted and CA paragraphs
are longer, so a pooled figure over-weights the easy domain.

In [ ]:
# Re-score the saved predictions from scratch, both scorers, split by domain.
preds = pd.read_csv(RESULTS / "predictions_sadeeddiac25_finetuned.csv", keep_default_na=False)

rows = []
for dom in ("CA", "MSA"):
    sub = preds[preds.domain == dom]
    r = [nfc(x) for x in sub.reference]; p = [nfc(x) for x in sub.prediction]
    row = {"domain": dom, "n": len(sub)}
    row.update(compute_der_wer_corrected(p, r)); row.update(compute_der_wer(p, r))
    rows.append(row)
t = pd.DataFrame(rows)
macro = {k: (round(v, 2) if isinstance(v, float) else v)
         for k, v in t.drop(columns=["domain", "n"]).mean().items()}
macro.update({"domain": "MACRO MEAN", "n": int(t.n.sum())})
t = pd.concat([t, pd.DataFrame([macro])], ignore_index=True)
t[["domain", "n", "DER_ce_corr", "DER_noce_corr", "WER_ce_corr", "WER_noce_corr"]]

### 9.2 The zero-shot baseline is a near-total failure, and that is a real finding

92.28 DER is not "bad diacritisation", it is **not attempting the task**. Two failure modes split
the benchmark almost in half:

- **604 of 1,200 predictions contain zero diacritics.** The model echoes the input back unchanged.
- The rest mostly produce meta-commentary in Arabic about what it is *going to* do, sometimes with
  markdown headings, rather than the diacritised text.

Mean diacritics per prediction: **2.5, against 159.7 in the reference.** 498 of 1,200 generations
also ran to the token cap.

The consequence for reading the headline: the 92.28 to 2.58 improvement is real but it is
measuring *instruction following plus the task*, not the task alone. At 0.8B the base model cannot
follow this instruction at all, which is exactly why full fine-tuning has so much room to work.

In [ ]:
b = pd.read_csv(RESULTS / "predictions_sadeeddiac25_baseline.csv", keep_default_na=False) \
    if (RESULTS / "predictions_sadeeddiac25_baseline.csv").exists() else None
if b is not None:
    DI = re.compile(r'[ً-ٰٟ]')
    b["pred_marks"] = b.prediction.apply(lambda s: len(DI.findall(s)))
    b["ref_marks"]  = b.reference.apply(lambda s: len(DI.findall(s)))
    print(f"predictions with ZERO diacritics: {(b.pred_marks == 0).sum()} / {len(b)}")
    print(f"mean marks per prediction: {b.pred_marks.mean():.1f}   "
          f"reference: {b.ref_marks.mean():.1f}")
else:
    print("baseline prediction dump not shipped (2.5 MB); numbers quoted above come from it")

### 9.3 The training curve, and which checkpoint shipped

One full epoch is 16,293 optimizer steps. `eval_loss` bottoms at **step 4,070**, one quarter of the
way in, and `load_best_model_at_end=True` restored those weights, so the shipped model is step
4,070 and `run_config.json` records that explicitly.

In [ ]:
ev = log.dropna(subset=["eval_loss"])[["step", "eval_loss"]].reset_index(drop=True)
tr = log.dropna(subset=["loss"])[["step", "loss"]]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tr.step, tr.loss, lw=.8, alpha=.4, label="train loss")
ax.plot(ev.step, ev.eval_loss, "o-", lw=2, label="eval loss")
best = ev.loc[ev.eval_loss.idxmin()]
ax.axvline(best.step, ls="--", c="crimson")
ax.annotate(f"best  step {int(best.step):,}\neval_loss {best.eval_loss:.5f}",
            (best.step, best.eval_loss), xytext=(14, 30), textcoords="offset points", c="crimson")
ax.set_xlabel("optimizer step"); ax.set_ylabel("loss"); ax.set_ylim(0, .5)
ax.legend(); ax.grid(alpha=.3)
ax.set_title("Qwen3.5-0.8B full FT: eval_loss bottoms at step 4,070 of 16,293")
plt.tight_layout(); plt.show()

ev.assign(rank=ev.eval_loss.rank().astype(int)).head(21)

`eval_loss` reaches its minimum a quarter of the way through the epoch and then oscillates in a
band roughly 0.117 to 0.153 for the remaining three quarters without recovering. Each example is
seen exactly once, so there is nothing to memorise and this is not classic overfitting; it reads
as the task saturating well before the corpus is exhausted.

**The practical reading: about three quarters of this run's GPU time bought nothing measurable.**
A 4,000 to 5,000 step rung would have produced the same shipped weights at roughly a quarter of
the cost. That is the single most actionable number in this notebook for planning the next run.

---
## 10. Findings

### 10.1 The smaller, fully fine-tuned model wins, and it is not close

Corrected scorer, SadeedDiac-25 macro mean, DER without case endings:

| | zero-shot | fine-tuned | relative reduction |
|---|---|---|---|
| Qwen3.5-4B + LoRA, 9.2% of the corpus | 59.57 | 8.48 | 85.8% |
| **Qwen3.5-0.8B full FT, one full epoch** | 92.28 | **2.58** | **97.2%** |

**2.58 against 8.48 is a 70% relative reduction at one fifth the parameter count.** The 0.8B
model also wins on every individual split, in both domains, under both scorers.

Two variables changed at once, model size and training method, so this run does not attribute the
win between them. The likely reading is that the method dominates: the 4B adapter trained 0.50%
of its weights on 9.2% of the corpus and its loss had plateaued by step 400, so it was not
capacity-limited. But a 0.8B LoRA rung or a 4B full-FT rung would be needed to prove that, and
neither was run.

### 10.2 The CA/MSA asymmetry survives full fine-tuning, smaller but the same shape

| | CA | MSA | ratio |
|---|---|---|---|
| 0.8B zero-shot | 92.60 | 91.95 | 0.99x |
| 4B + LoRA | 2.67 | 14.28 | 5.3x |
| **0.8B full FT** | **0.73** | **4.43** | **6.1x** |

Zero-shot the two domains are indistinguishable. After training they diverge sixfold, which is
if anything *wider* in relative terms than the 4B run.

The corpus composition explains it and `results/corpus_composition.csv` shows it directly: the
training corpus is overwhelmingly classical jurisprudence, hadith and tafsir. The largest single
source contributes about 49.5k rows and the top 25 files are all classical works; the only
recognisably modern sources (`aljazeera.txt`, `aljazeera-2016-12-29.b.txt`) contribute **216 rows
between them, about 0.02% of the corpus**.

So the finding from the 4B run holds and is now confirmed on a second model and a different
training method: **the asymmetry is created by the training data, not by the model or the
method**. A full epoch of a Classical-heavy corpus is a full epoch of more Classical Arabic, and
MSA error stays roughly 6x CA error no matter how much of it you train on.

This is why `load_training_data()` takes filename filters. Fixing this is a **data** change:

```python
train_raw, test_raw = load_training_data(exclude=r"<regex for the CA-heavy sources>")
```

No default mix is hardcoded, deliberately. Read `corpus_composition.csv` first, then choose.

### 10.3 Three quarters of the epoch bought nothing

`eval_loss` bottomed at step 4,070 of 16,293 and never recovered. The shipped weights are that
checkpoint, so the remaining 12,223 steps, roughly 3.5 hours of H100 time, contributed nothing to
the model that was actually kept.

Combined with the 4B run bottoming at step 400 of 1,500, the pattern is consistent across both
runs: **this task saturates early**, and step budget is not the binding constraint. Domain
balance is.

### 10.4 Measured cost and throughput

```
train_runtime           17,303.0 s   4h 48m 23s
train_steps_per_second       0.942   1.06 s/step
train_samples_per_second    60.261
total_flos                1.30e18
final train_loss           0.0235
```

`total_flos / runtime` is about **75 TFLOP/s** against an H100's 989 TFLOP/s bf16 peak, roughly
7.6% MFU. Low, and expected at this size: with a 248,320-token vocabulary the `lm_head` and its
fp32 cross-entropy upcast dominate, so the run is memory-bandwidth bound rather than
compute bound, and a 0.8B model has proportionally far less matmul to hide that behind than a 4B
one does. Disabling gradient checkpointing was still worth about 33%.

At the documented Modal rate of $4.58/hr for the GPU functions, the training run alone is about
**$22**, and the full pipeline including the CPU preparation steps, the smoke test and both
evaluation passes comes to roughly **$27**.

### 10.5 Caveats

- **The zero-shot baseline measures instruction following as much as diacritisation.** At 92.28
  DER the base model is not attempting the task; 604 of 1,200 outputs contain no diacritics at
  all. The 97.2% reduction is genuine but it is not 97.2% "better at diacritics".
- **The train-sample score is not a memorisation measure in the usual sense.** Every row was seen
  exactly once, and the shipped checkpoint is step 4,070, so it saw only the first quarter of the
  corpus.
- **The Tashkeela test score is partly a validation score.** The first 1,000 rows of the test
  split selected the checkpoint, so 1.91 is not an untouched held-out number.
- **This pipeline does not decontaminate.** The training split is loaded raw; the roughly 0.4%
  overlap figure comes from the dataset paper, not from a run.
- **Precision.** Aggregate DER reproduces to about +/-0.05 across identical runs, but only 81% of
  individual predictions are byte-identical, because greedy decoding is deterministic in principle
  and not bitwise on GPU. **One decimal is honest, two is not.** The 2.58 versus 8.48 gap is
  roughly 100x the noise floor, so it is solid.
- **Not comparable to the project spreadsheet.** The 6.49 (Tashkeel-350M-v2) and 5.15
  (gemma-4-E4B-it) bars were computed with the old scorer *and* without NFC, two axes at once.
  Beating 8.48 says this run beat the 4B LoRA run. It says nothing about those two until they are
  re-scored through this pipeline.

### 10.6 Recommended next steps

1. **Re-score Tashkeel-350M-v2 and gemma-4-E4B-it through this pipeline.** Until then it is
   unknown whether 2.58 clears the project's existing bar. Both are small and this is cheap.
2. **A domain-balanced rung at about 5,000 steps.** The two highest-value findings point the same
   way: MSA is the deficit, and the epoch saturates at a quarter of its length. A balanced-mix run
   capped near the saturation point costs about a quarter of this one.
3. **Do not spend on more steps or a bigger model** before doing the above. Neither was the
   binding constraint in either run.

---
## 11. Loading the fine-tuned model

Unlike the 4B LoRA run this is a **complete model**, not an adapter, so nothing else is needed.

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

path      = "./Qwen3.5-0.8B-Arabic-Diacritization-Full-FT-Best-Step-4070"
tokenizer = AutoTokenizer.from_pretrained(path)
model     = AutoModelForCausalLM.from_pretrained(path, dtype="auto", device_map="auto")
```

Generation must use the same settings the evaluation used, or the numbers do not transfer:
`enable_thinking=False` on the chat template, `eos_token_id=[248044, 248046]`, `do_sample=False`,
and `clean_output()` on the decoded string.

The weights are the best-validation checkpoint at step 4,070 (`eval_loss = 0.1072519422`), not
the final step. `run_config.json` inside the model folder records that provenance.